In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图基础风格
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接 """
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = "SELECT DISTINCT city FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact WHERE city IS NOT NULL AND city != ''"
    try:
        df = pd.read_sql(query, conn)
        return sorted(df['city'].tolist())
    finally:
        conn.close()

def fetch_price_data(city_name):
    """获取指定城市的全量总价数据"""
    conn = get_db_connection()
    query = "SELECT total_price FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact WHERE city = %s AND total_price IS NOT NULL"
    try:
        df = pd.read_sql(query, conn, params=(city_name,))
        return df['total_price'].values
    finally:
        conn.close()

In [ ]:
def plot_price_stairs(city):
    """核心绘图函数：阶梯填充图"""
    prices = fetch_price_data(city)
    
    if len(prices) == 0:
        return

    # --- 逻辑处理：自适应区间 ---
    # 剔除极高价位的离群值（取 0 到 98 分位数），防止长尾数据拉平图表
    upper_bound = np.percentile(prices, 98)
    filtered_prices = prices[prices <= upper_bound]
    
    # 计算直方图统计量：划分为 40 个阶梯
    counts, bins = np.histogram(filtered_prices, bins=40)

    # --- 绘图逻辑 ---
    fig, ax = plt.subplots(figsize=(12, 6))

    # 1. 绘制阶梯填充 
    ax.stairs(counts, bins, fill=True, color='#5DADE2', alpha=0.3, label='房源分布')
    
    # 2. 绘制阶梯边缘 
    ax.stairs(counts, bins, color='#2E86C1', linewidth=2)

    # 3. 标注中位数参考线
    median_val = np.median(prices)
    ax.axvline(median_val, color='#E74C3C', linestyle='--', alpha=0.8)
    ax.text(median_val, max(counts) * 0.9, f' 中位数: {median_val:.1f}万', 
            color='#E74C3C', fontweight='bold', va='top')

    # 图表修饰
    plt.title(f'[{city}] 二手房总价分布阶梯图', fontsize=15, pad=15)
    plt.xlabel('总价 (万元)', fontsize=11)
    plt.ylabel('房源数量 (套)', fontsize=11)
    sns.despine()
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [7]:
def init_dashboard():
    """初始化带筛选器的交互看板"""
    cities = fetch_cities()
    if not cities:
        return
    
    # 创建选择器，默认选中北京
    default_city = '北京' if '北京' in cities else cities[0]
    city_select = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='🏙️ 选择城市:',
        layout={'width': '250px'}
    )
    
    # 定义绘图输出区域
    output = widgets.Output()

    def update_chart(change):
        with output:
            clear_output(wait=True)
            plot_price_stairs(change['new'])

    city_select.observe(update_chart, names='value')

    # 初始渲染
    display(city_select, output)
    with output:
        plot_price_stairs(default_city)

In [8]:
if __name__ == "__main__":
    init_dashboard()

Dropdown(description='🏙️ 选择城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京',…

Output()